# EXP-EU-00 — Collecteur données Euronext EMA

**Objectif** : Collecter et valider tous les prix Euronext EMA (EUR/tonne) depuis 2010.
Ce notebook est la fondation incontournable avant toute étude Euronext.

## Checklist
1. Prix EMA quotidiens 2010–2025 — source + qualité
2. EUR/USD quotidien (conversion CBOT)
3. TTF gaz naturel EU (proxy coûts)
4. Corrélation EMA vs CBOT (validation du pivot)
5. Analyse des gaps / continuité des contrats
6. Saisonnalité EMA (différente du CBOT ?)

In [ ]:
import sys
from pathlib import Path

project_root = Path("../../..").resolve()
sys.path.insert(0, str(project_root / "src"))

import warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.dates as mdates

warnings.filterwarnings("ignore")
pd.set_option("display.float_format", "{:.4f}".format)

from mais.paths import RAW_DIR, INTERIM_DIR

## 1. Collecte des prix EMA

In [ ]:
# Lancer la collecte via le collecteur officiel
from mais.collect import run_collector

print("Collecte EMA...")
result = run_collector("euronext_ema")
print(f"  euronext_ema: {result}")

print("Collecte EUR/USD + TTF + BDI...")
result = run_collector("eu_cross_assets")
print(f"  eu_cross_assets: {result}")

In [ ]:
# Chargement et validation
ema_path = RAW_DIR / "euronext_ema" / "euronext_ema.csv"
eurusd_path = RAW_DIR / "eu_cross_assets" / "eu_cross_assets.csv"

ema = None
if ema_path.exists():
    ema = pd.read_csv(ema_path, parse_dates=["Date"])
    ema = ema.sort_values("Date").set_index("Date")
    print(f"EMA: {ema.shape}")
    print(f"  Période: {ema.index.min().date()} → {ema.index.max().date()}")
    print(f"  NaN ema_close: {ema['ema_close'].isna().sum()}")
    print(f"  Prix moyen: {ema['ema_close'].mean():.2f} EUR/t")
    print(f"  Prix min/max: {ema['ema_close'].min():.2f} / {ema['ema_close'].max():.2f} EUR/t")
else:
    print("⚠ EMA non collecté")
    print("Instructions manuelles:")
    print("  1. Aller sur https://www.euronext.com")
    print("  2. Products > Derivatives > Agricultural Commodities > Corn (EMA)")
    print("  3. Historical Data > Download CSV")
    print(f"  4. Sauvegarder dans: {ema_path}")
    print("  5. Colonnes attendues: Date, Open, High, Low, Close, Volume")

## 2. Corrélation EMA vs CBOT

In [ ]:
# Charger les prix CBOT
cbot_path = RAW_DIR / "cbot_corn" / "cbot_corn.csv"
eurusd_df = None

if eurusd_path.exists():
    eurusd_df = pd.read_csv(eurusd_path, parse_dates=["Date"]).set_index("Date")
    print(f"EUR/USD: {eurusd_df.shape}")

if ema is not None and cbot_path.exists() and eurusd_df is not None:
    cbot = pd.read_csv(cbot_path, parse_dates=["Date"]).set_index("Date")

    # Trouver la colonne close CBOT
    close_col = next((c for c in cbot.columns if "close" in c.lower()), cbot.columns[0])

    # Conversion CBOT USD/bu → EUR/tonne: ×39.368 / EUR_USD
    # 1 bushel corn = 25.4 kg → 1000 kg (tonne) = 1000/25.4 = 39.37 bu
    if "eurusd_rate" in eurusd_df.columns:
        cbot_eur_t = cbot[close_col] * 39.368 / eurusd_df["eurusd_rate"]
        merged = pd.DataFrame({
            "cbot_eur_t": cbot_eur_t,
            "ema_eur_t": ema["ema_close"],
        }).dropna()

        corr = merged.corr()
        print(f"\nCorrélation EMA vs CBOT (en EUR/t): {merged.corr().iloc[0, 1]:.4f}")
        print(f"Période commune: {merged.index.min().date()} → {merged.index.max().date()}")
        print(f"N jours: {len(merged)}")

        # Basis CBOT-EMA
        basis = cbot_eur_t - ema["ema_close"]
        basis_valid = basis.dropna()
        print(f"\nBasis CBOT-EMA (EUR/t):")
        print(f"  Moyen: {basis_valid.mean():.2f}")
        print(f"  Std:   {basis_valid.std():.2f}")
        print(f"  Min/Max: {basis_valid.min():.2f} / {basis_valid.max():.2f}")
    else:
        print("EUR/USD non disponible pour la conversion")
else:
    print("⚠ Données insuffisantes pour la corrélation")
    print("  Collecter d'abord: euronext_ema, eu_cross_assets, cbot_corn")

## 3. Qualité et continuité des données

In [ ]:
if ema is not None:
    # Analyser les gaps
    date_diffs = ema.index.to_series().diff().dt.days.dropna()
    big_gaps = date_diffs[date_diffs > 5]

    print(f"Gaps > 5 jours calendaires: {len(big_gaps)}")
    if not big_gaps.empty:
        print("  Principaux gaps:")
        for dt, n_days in big_gaps.nlargest(5).items():
            print(f"    {dt.date()}: {int(n_days)} jours")

    # Analyse annuelle
    annual = ema["ema_close"].groupby(ema.index.year).agg(["count", "mean", "std"])
    annual.columns = ["n_jours", "prix_moyen_eur_t", "std"]
    print("\nAnalyse annuelle EMA:")
    print(annual.to_string(float_format="{:.2f}".format))
else:
    print("EMA non disponible")

In [ ]:
if ema is not None:
    fig, axes = plt.subplots(2, 2, figsize=(16, 10))

    # Prix EMA historique
    ax = axes[0, 0]
    ema["ema_close"].plot(ax=ax, color="tomato", linewidth=0.8)
    ax.set_title("Prix Euronext EMA (EUR/tonne)")
    ax.set_ylabel("EUR/tonne")
    ax.xaxis.set_major_formatter(mdates.DateFormatter("%Y"))
    ax.grid(alpha=0.3)

    # Distribution des retours journaliers
    ax = axes[0, 1]
    returns = ema["ema_close"].pct_change().dropna()
    ax.hist(returns, bins=100, color="tomato", alpha=0.7, edgecolor="white")
    ax.axvline(returns.mean(), color="black", linestyle="--", label=f"Moyenne: {returns.mean():.4f}")
    ax.set_title("Distribution retours journaliers EMA")
    ax.set_xlabel("Retour journalier")
    ax.legend()
    ax.grid(alpha=0.3)

    # Saisonnalité mensuelle
    ax = axes[1, 0]
    monthly_returns = returns.groupby(returns.index.month).mean()
    ax.bar(range(1, 13), monthly_returns * 100, color="tomato", alpha=0.7)
    ax.set_xticks(range(1, 13))
    ax.set_xticklabels(["J", "F", "M", "A", "M", "J", "J", "A", "S", "O", "N", "D"])
    ax.axhline(0, color="black", linewidth=0.8)
    ax.set_title("Retour mensuel moyen EMA (%)")
    ax.set_ylabel("%")
    ax.grid(alpha=0.3, axis="y")

    # EMA vs CBOT comparaison si disponible
    ax = axes[1, 1]
    if 'merged' in locals() and not merged.empty:
        merged_norm = merged / merged.iloc[0] * 100  # base 100
        merged_norm["cbot_eur_t"].plot(ax=ax, label="CBOT (converti EUR/t)", color="steelblue", linewidth=0.8)
        merged_norm["ema_eur_t"].plot(ax=ax, label="EMA (EUR/t)", color="tomato", linewidth=0.8)
        ax.set_title("EMA vs CBOT (base 100, en EUR/t)")
        ax.legend()
        ax.grid(alpha=0.3)
    else:
        ax.text(0.5, 0.5, "CBOT ou EUR/USD\nnon disponible",
                ha="center", va="center", transform=ax.transAxes, fontsize=12)

    plt.suptitle("Analyse données Euronext EMA", fontsize=14, fontweight="bold")
    plt.tight_layout()
    plt.savefig(project_root / "artefacts" / "ema_data_quality.png", dpi=150, bbox_inches="tight")
    plt.show()
else:
    print("⚠ Visualisation impossible sans données EMA")

## 4. Conclusion et checklist

In [ ]:
print("=" * 60)
print("CHECKLIST EXP-EU-00")
print("=" * 60)

checks = {
    "Prix EMA disponibles": ema is not None,
    "EUR/USD disponible": eurusd_df is not None and not eurusd_df.empty,
    "EMA ≥ 500 jours de données": ema is not None and len(ema) >= 500,
    "EMA ≥ 2010": ema is not None and ema.index.min().year <= 2012,
    "Corrélation calculée": 'corr' in dir() or 'merged' in dir(),
}

all_ok = True
for check, ok in checks.items():
    symbol = "✓" if ok else "✗"
    print(f"  {symbol} {check}")
    if not ok:
        all_ok = False

print()
if all_ok:
    print("→ FONDATIONS EMA OK — Prêt pour EXP-EU-00B (benchmark pivot)")
else:
    print("→ FONDATIONS INCOMPLÈTES — Résoudre les points ✗ avant le benchmark")